# Sample Sociodemographic EDA

This notebook describes the sociodemographic composition of the sample of 100 personas used to generate LLM networks, and shows how key characteristics vary by ego political affiliation.


## 1. Setup

Load the persona samples and define plotting helpers. Percentages are unweighted by default because the generated networks use sampled personas as nodes. Weighted percentages are computed as a diagnostic using `weight_post`.


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

SAMPLE_FILES = {
    "sample 100": "data/anes_100.csv",
}

FOCUS_SAMPLE = "sample 100"
WEIGHT_COL = "weight_post"

SOCIODEMOGRAPHIC_VARS = {
    "profile_gender": "Gender",
    "profile_age_bucket": "Age",
    "profile_racethnicity": "Race/ethnicity",
    "profile_educ5": "Education",
    "profile_region4": "Region",
    "profile_relig": "Religion",
    "w2presvtwho": "2020 vote / affiliation",
}

ORDERED_CATEGORIES = {
    "profile_gender": ["female", "male"],
    "profile_age_bucket": ["18-29", "30-44", "45-59", "60-74", "75+"],
    "profile_racethnicity": ["white", "black", "hispanic", "ethnic minority"],
    "profile_educ5": [
        "less than high school",
        "high school graduate",
        "vocational tech school",
        "bachelor's degree",
        "postgraduate",
    ],
    "profile_region4": ["Northeast", "Midwest", "South", "West"],
    "profile_relig": [
        "Protestant",
        "Roman Catholic",
        "just Christian",
        "believer in a minority religion",
        "not religious",
        "agnostic",
        "atheist",
    ],
    "w2presvtwho": ["Democrat", "Republican", "Independent", "didn't vote"],
}

EGO_PARTY_ORDER = ["dem", "rep", "other"]
EGO_PARTY_LABELS = {"dem": "Dem ego", "rep": "Rep ego", "other": "Other ego"}
EGO_PARTY_PALETTE = {"Dem ego": "#2f6fdd", "Rep ego": "#c43c39", "Other ego": "#4BA05C"}
SAMPLE_PALETTE = {"sample 100": "#b14747"}


In [ ]:
def normalize_party(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower()
    if text in {"democrat", "joe biden", "biden"}:
        return "dem"
    if text in {"republican", "donald trump", "trump"}:
        return "rep"
    return "other"


def ordered_values(df, var):
    observed = [x for x in df[var].dropna().unique().tolist()]
    preferred = ORDERED_CATEGORIES.get(var, [])
    ordered = [x for x in preferred if x in observed]
    ordered += sorted([x for x in observed if x not in ordered])
    return ordered


def categorical_distribution(df, var, group_col="sample", weight_col=None):
    d = df[[group_col, var] + ([weight_col] if weight_col else [])].dropna(subset=[var]).copy()
    if weight_col:
        counts = (
            d.groupby([group_col, var], dropna=False)[weight_col]
            .sum()
            .rename("count")
            .reset_index()
        )
    else:
        counts = (
            d.groupby([group_col, var], dropna=False)
            .size()
            .rename("count")
            .reset_index()
        )
    counts["prop"] = counts["count"] / counts.groupby(group_col)["count"].transform("sum")
    return counts


def plot_sample_distribution(df, var, weighted=False, ax=None):
    weight_col = WEIGHT_COL if weighted else None
    plot_df = categorical_distribution(df, var, weight_col=weight_col)
    order = ordered_values(df, var)
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 4))
    sns.barplot(
        data=plot_df,
        x=var,
        y="prop",
        hue="sample",
        order=order,
        palette=SAMPLE_PALETTE,
        ax=ax,
    )
    ax.set_title(f"{SOCIODEMOGRAPHIC_VARS[var]}")
    ax.set_xlabel(SOCIODEMOGRAPHIC_VARS[var])
    ax.set_ylabel("Share of sample")
    ax.tick_params(axis="x", rotation=35)
    return ax


def plot_by_ego_party(df, var, sample=FOCUS_SAMPLE, ax=None):
    sub = df[df["sample"] == sample].copy()
    sub = sub[sub["ego_party"].isin(EGO_PARTY_ORDER)].dropna(subset=[var])
    counts = (
        sub.groupby(["ego_party", var], dropna=False)
        .size()
        .rename("count")
        .reset_index()
    )
    counts["prop"] = counts["count"] / counts.groupby("ego_party")["count"].transform("sum")
    counts["ego_party_label"] = counts["ego_party"].map(EGO_PARTY_LABELS)
    order = ordered_values(sub, var)
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 4))
    sns.barplot(
        data=counts,
        x=var,
        y="prop",
        hue="ego_party_label",
        hue_order=[EGO_PARTY_LABELS[p] for p in EGO_PARTY_ORDER],
        order=order,
        palette=EGO_PARTY_PALETTE,
        ax=ax,
    )
    ax.set_title(f"{SOCIODEMOGRAPHIC_VARS[var]} by ego affiliation ({sample})")
    ax.set_xlabel(SOCIODEMOGRAPHIC_VARS[var])
    ax.set_ylabel("Share within ego affiliation")
    ax.tick_params(axis="x", rotation=35)
    return ax


## 2. Load Samples

Each row is one ANES respondent/persona used as a graph node. `ego_party` is derived from `w2presvtwho` as Democrat, Republican, or other.


In [ ]:
samples = []
for sample_name, file_name in SAMPLE_FILES.items():
    tmp = pd.read_csv(file_name)
    tmp["sample"] = sample_name
    samples.append(tmp)

sample_df = pd.concat(samples, ignore_index=True)
sample_df["ego_party"] = sample_df["w2presvtwho"].apply(normalize_party)
sample_df["ego_party_label"] = sample_df["ego_party"].map(EGO_PARTY_LABELS)

print(sample_df.groupby("sample").size().rename("n"))
sample_df.head()


In [ ]:
sample_overview = (
    sample_df
    .groupby("sample")
    .agg(
        n=("caseid", "size"),
        mean_weight=(WEIGHT_COL, "mean"),
        sd_weight=(WEIGHT_COL, "std"),
        min_age=("profile_age", "min"),
        mean_age=("profile_age", "mean"),
        max_age=("profile_age", "max"),
    )
    .reset_index()
)

sample_overview.round(3)


## 3. Sample Composition

These bar charts show the main sociodemographic variables for sample 100. They are unweighted, matching the node composition used in generated networks.


In [ ]:
plot_vars = [
    "profile_gender",
    "profile_age_bucket",
    "profile_racethnicity",
    "profile_educ5",
    "profile_region4",
    "profile_relig",
    "w2presvtwho",
]

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.ravel()
for ax, var in zip(axes, plot_vars):
    plot_sample_distribution(sample_df, var, weighted=False, ax=ax)
    ax.legend(title="Sample")
    sns.despine(ax=ax)
for ax in axes[len(plot_vars):]:
    ax.axis("off")
plt.tight_layout()
plt.show()


## 5. State Composition

State is high-cardinality, so this section shows the largest states in sample 100.


In [ ]:
state_counts = (
    sample_df
    .groupby(["sample", "profile_state"], dropna=False)
    .size()
    .rename("count")
    .reset_index()
)
state_counts["prop"] = state_counts["count"] / state_counts.groupby("sample")["count"].transform("sum")

top_states = (
    state_counts
    .sort_values(["sample", "count"], ascending=[True, False])
    .groupby("sample")
    .head(15)
)

g = sns.catplot(
    data=top_states,
    y="profile_state",
    x="prop",
    hue="sample",
    col="sample",
    kind="bar",
    sharey=False,
    height=5,
    aspect=.9,
    palette=SAMPLE_PALETTE,
)
g.set_axis_labels("Share of sample", "State")
g.set_titles("{col_name}")
g.fig.suptitle("Top states in sample 100", y=1.03)
plt.show()

top_states.sort_values(["sample", "count"], ascending=[True, False]).head(30)


## 6. Composition by Ego Political Affiliation

These charts describe sample 100 by ego political affiliation.


In [ ]:
focus_df = sample_df[sample_df["sample"] == FOCUS_SAMPLE].copy()

ego_counts = (
    focus_df["ego_party_label"]
    .value_counts(dropna=False)
    .rename_axis("ego_party")
    .reset_index(name="count")
)
ego_counts["prop"] = ego_counts["count"] / ego_counts["count"].sum()

plt.figure(figsize=(6, 4))
sns.barplot(
    data=ego_counts,
    x="ego_party",
    y="prop",
    order=[EGO_PARTY_LABELS[p] for p in EGO_PARTY_ORDER],
    palette=EGO_PARTY_PALETTE,
)
plt.title(f"Ego political affiliation ({FOCUS_SAMPLE})")
plt.xlabel("Ego affiliation")
plt.ylabel("Share of sample")
plt.tight_layout()
plt.show()

ego_counts


In [ ]:
ego_plot_vars = [
    "profile_gender",
    "profile_age_bucket",
    "profile_racethnicity",
    "profile_educ5",
    "profile_region4",
    "profile_relig",
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()
for ax, var in zip(axes, ego_plot_vars):
    plot_by_ego_party(sample_df, var, sample=FOCUS_SAMPLE, ax=ax)
    ax.legend(title="Ego affiliation")
    sns.despine(ax=ax)
plt.tight_layout()
plt.show()


## 7. Exportable Tables

Compact tables for reporting sample composition in the paper or appendix.


In [ ]:
composition_tables = []
for var, label in SOCIODEMOGRAPHIC_VARS.items():
    dist = categorical_distribution(sample_df, var, weight_col=None)
    dist["variable"] = label
    dist = dist.rename(columns={var: "category"})
    composition_tables.append(dist[["sample", "variable", "category", "count", "prop"]])

composition_table = pd.concat(composition_tables, ignore_index=True)

composition_table.head(20)


In [ ]:
ego_composition_tables = []
for var, label in SOCIODEMOGRAPHIC_VARS.items():
    sub = sample_df[sample_df["sample"] == FOCUS_SAMPLE].copy()
    sub = sub[sub["ego_party"].isin(EGO_PARTY_ORDER)].dropna(subset=[var])
    counts = (
        sub.groupby(["ego_party", var], dropna=False)
        .size()
        .rename("count")
        .reset_index()
    )
    counts["prop"] = counts["count"] / counts.groupby("ego_party")["count"].transform("sum")
    counts["variable"] = label
    counts["sample"] = FOCUS_SAMPLE
    counts["ego_party_label"] = counts["ego_party"].map(EGO_PARTY_LABELS)
    counts = counts.rename(columns={var: "category"})
    ego_composition_tables.append(counts[["sample", "ego_party", "ego_party_label", "variable", "category", "count", "prop"]])

ego_composition_table = pd.concat(ego_composition_tables, ignore_index=True)

ego_composition_table.head(20)


## 8. Notes

- The unweighted plots describe the actual node composition used in the LLM-generated networks.
- The weighted plots are useful for checking how the sample compares to the ANES weighted target population, but they do not change the generated network node counts.
- `other` ego affiliation includes independents and non-voters because the network validation distinguishes Democratic and Republican exposure targets from ego party categories.


In [ ]:
# =====================================================
# TABLE S1: SAMPLE COMPOSITION
# =====================================================

TABLE_VARS = {
    "profile_gender": "Gender",
    "profile_age_bucket": "Age",
    "profile_racethnicity": "Race/ethnicity",
    "profile_educ5": "Education",
    "profile_region4": "Region",
    "profile_relig": "Religion",
    "w2presvtwho": "2020 vote",
}

ORDERED_CATEGORIES = {
    "profile_gender": ["female", "male"],
    "profile_age_bucket": ["18-29", "30-44", "45-59", "60-74", "75+"],
    "profile_racethnicity": [
        "white", "black", "hispanic", "ethnic minority"
    ],
    "profile_educ5": [
        "less than high school",
        "high school graduate",
        "vocational tech school",
        "bachelor's degree",
        "postgraduate",
    ],
    "profile_region4": [
        "Northeast", "Midwest", "South", "West"
    ],
    "profile_relig": [
        "Protestant",
        "Roman Catholic",
        "just Christian",
        "believer in a minority religion",
        "not religious",
        "agnostic",
        "atheist",
    ],
    "w2presvtwho": [
        "Democrat", "Republican", "Independent", "didn't vote"
    ],
}


def make_sample_comparison_table(full_df, sample_df, weight_col="weight_post"):

    rows = []

    for var, label in TABLE_VARS.items():

        # Weighted distribution in eligible ANES population
        full = (
            full_df.groupby(var)[weight_col]
            .sum()
        )
        full = full / full.sum()

        # Counts and unweighted distribution in N=100 sample
        sample_n = sample_df[var].value_counts()
        sample_prop = sample_n / sample_n.sum()

        categories = ORDERED_CATEGORIES[var]

        for category in categories:
            if category not in full.index and category not in sample_prop.index:
                continue

            full_pct = 100 * full.get(category, 0)
            n = int(sample_n.get(category, 0))
            sample_pct = 100 * sample_prop.get(category, 0)

            rows.append({
                "Characteristic": label,
                "Category": category,
                "Weighted ANES (%)": full_pct,
                "N=100 (n)": n,
                "N=100 (%)": sample_pct,
                "Difference (pp)": sample_pct - full_pct,
            })

    table = pd.DataFrame(rows)

    # Only show the characteristic name on the first row of each block
    table["Characteristic"] = (
        table["Characteristic"]
        .where(~table["Characteristic"].duplicated(), "")
    )

    return table


# df is the eligible ANES population produced by get_personas()
# samples[100] is the sampled population used in the networks
table_s1 = make_sample_comparison_table(
    full_df=sample_df,
    sample_df=samples[100],
)

# Show table
display(
    table_s1.style.format({
        "Weighted ANES (%)": "{:.1f}",
        "N=100 (%)": "{:.1f}",
        "Difference (pp)": "{:+.1f}",
    })
)

In [ ]:
sample_df